In [8]:
# ======================= 1. Imports & Config ========================
import os, glob, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "/kaggle/input/brats2015/BRATS2015/training"
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# User hyperparams for ~2 hour run
NUM_CASES = 80   # ~60 random patients from BRATS2015 training set (adjust for 2h budget)
PATCH_SIZE = (64, 64, 64)
BATCH_SIZE = 2
EPOCHS = 160
NUM_WORKERS = 2  # Increase if you have lots of CPU/RAM; 2 is safe for Kaggle
BASE_FILTERS = 16

NUM_CLASSES = 5
NUM_MODALITIES = 4
FUSION_WEIGHTS = torch.tensor([0.25, 0.25, 0.25, 0.25], dtype=torch.float32)  # Set/fetch dynamically if available

print('Device:', DEVICE)

Device: cuda


In [9]:
# ==================== 2. Utility: MRI Loading =====================
import SimpleITK as sitk
def load_patient_mha(folder):
    # Auto-find files for all expected modalities in one patient folder
    files = sorted(glob.glob(os.path.join(folder, "*.mha")))
    t1 = next(f for f in files if "t1." in f.lower() and not "t1c" in f.lower())
    t1ce = next(f for f in files if "t1c" in f.lower())
    t2 = next(f for f in files if "t2." in f.lower())
    flair = next(f for f in files if "flair" in f.lower())
    seg = next(f for f in files if "ot." in f.lower())
    # Load and convert to numpy [C, D, H, W]
    vols = []
    for f in [t1, t2, t1ce, flair]:
        img = sitk.GetArrayFromImage(sitk.ReadImage(f)).astype(np.float32)
        # Normalize each modality to zero mean/unit std (brain only)
        img_mask = img > 0
        img_mean = img[img_mask].mean() if img_mask.sum() else img.mean()
        img_std = img[img_mask].std() if img_mask.sum() else img.std()
        img = (img - img_mean) / (img_std + 1e-5)
        vols.append(img)
    vol = np.stack(vols, axis=0)  # [4, D, H, W]
    seg = sitk.GetArrayFromImage(sitk.ReadImage(seg)).astype(np.int16)  # [D, H, W]
    return vol, seg

In [10]:
# ============= 3. Simple PatchSampler Dataset (BRATS subsample) =============
class BRATSPatchDataset(Dataset):
    def __init__(self, root, n_cases=40, patch=(64,64,64), patches_per_case=5, fusion_weights=FUSION_WEIGHTS):
        folders = glob.glob(os.path.join(root, "*/*/"))
        random.shuffle(folders)
        self.cases = folders[:n_cases]
        self.volumes = []
        self.labels = []
        for case in tqdm(self.cases, desc="Loading cases"):
            try:
                v, l = load_patient_mha(case)
                self.volumes.append(v)
                self.labels.append(l)
            except Exception as e:
                print(f"Skipping {case}: {e}")
        self.patch = patch
        self.patches_per_case = patches_per_case
        self.fusion_weights = fusion_weights
        print("Loaded:", len(self.volumes))
    def __len__(self):
        return len(self.volumes) * self.patches_per_case
    def __getitem__(self, idx):
        case_idx = idx // self.patches_per_case
        img4, seg = self.volumes[case_idx], self.labels[case_idx]
        D, H, W = img4.shape[1:]
        pd, ph, pw = self.patch
        z0 = np.random.randint(0, D - pd)
        y0 = np.random.randint(0, H - ph)
        x0 = np.random.randint(0, W - pw)
        patch4 = img4[:, z0:z0+pd, y0:y0+ph, x0:x0+pw]
        patch4 = torch.from_numpy(patch4).float()           # convert to torch
        fw = self.fusion_weights.to(patch4.device) if isinstance(self.fusion_weights, torch.Tensor) else torch.tensor(self.fusion_weights, dtype=torch.float32)
        fused = (patch4 * fw.view(4,1,1,1)).sum(dim=0, keepdim=True)
        label_patch = torch.from_numpy(seg[z0:z0+pd, y0:y0+ph, x0:x0+pw]).long()
        return fused, label_patch

In [11]:
# =========== 4. ResUNet3D Model =============
class ResidualBlock3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        identity = self.proj(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, in_ch, 3, stride=2, padding=1)
        self.rb = ResidualBlock3d(in_ch, out_ch)
    def forward(self, x):
        x = self.down(x)
        return self.rb(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.rb = ResidualBlock3d(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        # Automatic spatial padding
        ds = [skip.size(d) - x.size(d) for d in range(2,5)]
        x = F.pad(x, [0, ds[2], 0, ds[1], 0, ds[0]]) if any(ds) else x
        x = torch.cat((x, skip), 1)
        return self.rb(x)

class ResUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=5, base=BASE_FILTERS):
        super().__init__()
        self.stem = ResidualBlock3d(in_ch, base)
        self.down1 = DownBlock(base, base*2)
        self.down2 = DownBlock(base*2, base*4)
        self.down3 = DownBlock(base*4, base*8)
        self.down4 = DownBlock(base*8, base*16)
        self.up1 = UpBlock(base*16, base*8, base*8)
        self.up2 = UpBlock(base*8, base*4, base*4)
        self.up3 = UpBlock(base*4, base*2, base*2)
        self.up4 = UpBlock(base*2, base, base)
        self.head = nn.Conv3d(base, out_ch, 1)
    def forward(self, x):
        s1 = self.stem(x)
        d1 = self.down1(s1)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        u1 = self.up1(d4, d3)
        u2 = self.up2(u1, d2)
        u3 = self.up3(u2, d1)
        u4 = self.up4(u3, s1)
        return self.head(u4)

In [12]:
# ============ 5. Focal Loss For Multiclass =============
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        n, c, d, h, w = logits.shape
        logits = logits.view(n, c, -1)
        targets = targets.view(n, -1)
        logpt = F.log_softmax(logits, dim=1)
        pt = torch.exp(logpt)
        at = self.alpha.unsqueeze(0).unsqueeze(2)
        targets_oh = F.one_hot(targets, num_classes=c).permute(0, 2, 1).float()
        loss = -at * (1 - pt) ** self.gamma * logpt * targets_oh
        return loss.sum(dim=1).mean()
# Class weights for class imbalance: tune as needed
ALPHA = torch.tensor([1, 3, 3, 4, 6], dtype=torch.float32).to(DEVICE) # 0:normal...4:enh

In [13]:
# ============= 6. Metrics ============
def dice_score(pred, target, num_classes=5):
    with torch.no_grad():
        pred = torch.argmax(pred, dim=1)
        dices = []
        for c in range(num_classes):
            p = (pred == c)
            t = (target == c)
            inter = (p & t).sum().item()
            den = p.sum().item() + t.sum().item()
            dices.append((2 * inter + 1e-4) / (den + 1e-4))
        return np.mean(dices)

In [14]:
# ============= 7. Run Training Loop ==========================
print("Setting up datasets and model...")
train_ds = BRATSPatchDataset(DATA_ROOT, n_cases=NUM_CASES, patch=PATCH_SIZE, patches_per_case=6)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
model = ResUNet3D(in_ch=1, out_ch=5).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = FocalLoss(ALPHA, gamma=2)

print("Training...")
for epoch in range(EPOCHS):
    model.train()
    losses = []
    dices = []
    for x, y in tqdm(train_dl, total=len(train_dl), leave=False, desc=f"Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        dices.append(dice_score(out, y))
    mean_loss = np.mean(losses)
    mean_dice = np.mean(dices)
    print(f"Epoch {epoch+1:3d}: loss={mean_loss:.3f} dice={mean_dice:.3f}")

print("Training complete.")

Setting up datasets and model...


Loading cases: 100%|██████████| 80/80 [00:41<00:00,  1.93it/s]


Loaded: 80
Training...


Epoch   1: loss=0.754 dice=0.174


Epoch   2: loss=0.178 dice=0.474


...

Epoch 156: loss=0.079 dice=0.558


Epoch 157: loss=0.076 dice=0.610


Epoch 158: loss=0.087 dice=0.590


Epoch 159: loss=0.072 dice=0.612


Epoch 160: loss=0.074 dice=0.572
Training complete.
